# Fase 2: Filtros y Reducción de Dimensionalidad
En este cuaderno tomaremos la base de datos explorada y construiremos cuatro escenarios distintos de características (features) para evaluar posteriormente cuál ofrece el mejor rendimiento predictivo.

1. **Escenario 1 (Crudo):** Todas las características originales estandarizadas.
2. **Escenario 2 (Filtro Manual):** Eliminación de variables con alta colinealidad (>0.85) o bajo poder predictivo (<0.05).
3. **Escenario 3 (Selección LASSO):** Selección algorítmica incrustada usando penalización L1.
4. **Escenario 4 (PCA):** Reducción de dimensionalidad extrayendo los componentes principales que expliquen el 95% de la varianza.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

## 1. Carga de Datos y Estandarización Base (Escenario 1)
Para esta fase de transformación, partimos de la **base de datos original e intacta** (cargada desde nuestro archivo local `dataset_cancer.csv` para agilizar el proceso). 

Dado que los algoritmos que utilizaremos para la selección y reducción (LASSO y PCA) son altamente sensibles a la magnitud geométrica de los datos, es **estrictamente necesario estandarizar** todas las variables (Z-score) antes de aplicar cualquier método. Esta matriz estandarizada completa será nuestro Escenario 1.

In [2]:
# Cargar el dataset desde la carpeta local
df = pd.read_csv('./data/dataset_cancer.csv')

# Separar variables independientes (X) y el objetivo (y)
X = df.drop('Diagnosis', axis=1)
y = df['Diagnosis']

# Estandarización (Z-score)
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)

# Convertir de vuelta a DataFrame para no perder los nombres de las columnas
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)

# Renombrar conceptualmente para claridad del proyecto
X_escenario_1 = X_scaled 

print("Datos cargados y estandarizados.")
print(f"Shape del Escenario 1 (Datos Crudos): {X_escenario_1.shape}")

Datos cargados y estandarizados.
Shape del Escenario 1 (Datos Crudos): (569, 30)


## 2. Escenario 2: Filtro Manual (Heurístico)
Basados en el Análisis Exploratorio (EDA) del cuaderno anterior, implementamos un algoritmo que evalúa la multicolinealidad y el poder predictivo simultáneamente:
1. `umbral_colinealidad = 0.85`: Si dos variables están altamente correlacionadas, se elimina aquella que tenga menor correlación con el diagnóstico (la variable objetivo).
2. `umbral_ruido = 0.05`: Si una variable tiene una correlación casi nula con el diagnóstico, se descarta por ser considerada ruido estadístico.

In [3]:
# 1. Definir umbrales matemáticos
umbral_colinealidad = 0.85  # Si dos variables se parecen más de esto, una debe irse
umbral_ruido = 0.05         # Si una variable tiene menos de esto de poder predictivo, es basura

# 2. Obtener el poder predictivo (Correlación absoluta con Diagnosis)
# Unimos temporalmente X_scaled e y para calcular su correlación
df_temp = pd.concat([X_scaled, y.reset_index(drop=True)], axis=1)
corr_objetivo = df_temp.corr()['Diagnosis'].drop('Diagnosis').abs()

# 3. Calcular la matriz de correlación solo entre las predictoras (absoluta)
matriz_corr_X = X_scaled.corr().abs()

# Lista para guardar las variables condenadas a ser eliminadas
columnas_a_eliminar = set()

# 4. ALGORITMO DE ELIMINACIÓN POR MULTICOLINEALIDAD
# Recorrer el triángulo superior de la matriz para evaluar cada par una sola vez
for i in range(len(matriz_corr_X.columns)):
    for j in range(i+1, len(matriz_corr_X.columns)):
        var1 = matriz_corr_X.columns[i]
        var2 = matriz_corr_X.columns[j]
        
        # Si la correlación entre ellas supera el umbral y ninguna ha sido eliminada aún
        if matriz_corr_X.iloc[i, j] > umbral_colinealidad:
            # Eliminar la que tenga MENOR poder predictivo con el diagnóstico
            if corr_objetivo[var1] < corr_objetivo[var2]:
                columnas_a_eliminar.add(var1)
            else:
                columnas_a_eliminar.add(var2)

# 5. ALGORITMO DE ELIMINACIÓN POR RUIDO ESTADÍSTICO
for col in X_scaled.columns:
    if corr_objetivo[col] < umbral_ruido:
        columnas_a_eliminar.add(col)

columnas_a_eliminar = list(columnas_a_eliminar)

# 6. Ejecutar la limpieza para crear el Escenario 2
X_manual = X_scaled.drop(columns=columnas_a_eliminar)

print(f"Dimensiones de Escenario 1 (Crudo): {X_scaled.shape}")
print(f"Dimensiones de Escenario 2 (Filtro Manual): {X_manual.shape}")
print(f"Cantidad de variables eliminadas: {len(columnas_a_eliminar)}\n")

print("=== VARIABLES ELIMINADAS ===")
for col in sorted(columnas_a_eliminar):
    print(f"- {col}")

Dimensiones de Escenario 1 (Crudo): (569, 30)
Dimensiones de Escenario 2 (Filtro Manual): (569, 14)
Cantidad de variables eliminadas: 16

=== VARIABLES ELIMINADAS ===
- area1
- area2
- area3
- compactness1
- compactness3
- concave_points1
- concavity1
- concavity3
- fractal_dimension1
- perimeter1
- perimeter2
- radius1
- radius3
- symmetry2
- texture1
- texture2


### Justificación Teórica: Sinergia entre Selección de Características y Modelos Predictivos

Para reducir el espacio dimensional de 30 variables a un subconjunto óptimo, hemos diseñado una estrategia basada en la naturaleza matemática de los clasificadores que utilizaremos en la Fase 3. 

Se evaluaron diversas técnicas de la literatura, descartando enfoques univariados y categóricos por las siguientes razones clínicas y matemáticas:

1. **Descarte de Chi-Cuadrado ($\chi^2$):** Esta técnica está diseñada para evaluar frecuencias y variables categóricas. Dado que nuestras mediciones geométricas celulares (radio, textura, área) son variables continuas, forzar una prueba $\chi^2$ mediante escalado artificial comprometería la validez estadística del análisis.
2. **Descarte de ANOVA (F-Test) a favor de LASSO:** Aunque ANOVA F-test es un método de filtro válido para variables continuas, es estrictamente **univariado**. Es decir, evalúa el poder predictivo de cada característica de forma aislada, ignorando la redundancia. Por ello, hemos optado por utilizar **LASSO (Regularización L1)**, el cual es un método integrado (*Embedded Method*) que no solo evalúa el poder predictivo lineal, sino que aborda la multicolinealidad de forma **multivariada**, "apagando" coeficientes redundantes simultáneamente.
3. **El Contraste (Lineal vs. No Lineal):** Somos conscientes de que LASSO (al igual que la Correlación de Pearson) asume relaciones lineales. Extraerá las mejores variables para alimentar modelos lineales (como la Regresión Logística), pero podría descartar variables que posean relaciones complejas o curvas con el diagnóstico. 

Por esta razón, nuestra arquitectura de escenarios de datos contemplará dos grandes campeones de selección:
* **Escenario LASSO:** Diseñado para optimizar nuestro modelo de Regresión Logística.
* **Escenario Mutual Information (Información Mutua):** Se aplicará este filtro agnóstico y no lineal para garantizar que algoritmos complejos (como las Redes Neuronales y Random Forest) reciban las características con relaciones no lineales que necesitan para construir sus fronteras de decisión abstractas.

## 3. Escenario 3: Selección Incrustada con LASSO
En lugar de filtrar matemáticamente variable por variable, delegaremos la tarea a la regularización **L1 (LASSO)**. Entrenaremos una Regresión Logística con un nivel de penalización estricto (`C=1`) y usaremos `SelectFromModel` para extraer únicamente las variables cuyo coeficiente matemático haya sobrevivido (sea distinto de 0).

In [4]:
# Configurar la Regresión Logística con penalización LASSO (L1)
modelo_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=1, random_state=30)

# Selector de características basado en el modelo
selector = SelectFromModel(modelo_lasso)
selector.fit(X_scaled, y)

# Obtener los nombres de las variables que sobrevivieron
variables_lasso = X_scaled.columns[selector.get_support()]

# Crear el Escenario 3
X_lasso = X_scaled[variables_lasso]

print(f"Variables descartadas por LASSO: {X_scaled.shape[1] - len(variables_lasso)}")
print(f"Variables sobrevivientes ({len(variables_lasso)}):")
for var in variables_lasso:
    print(f"- {var}")
    
print(f"\nShape del Escenario 3 (LASSO): {X_lasso.shape}")

Variables descartadas por LASSO: 15
Variables sobrevivientes (15):
- concavity1
- concave_points1
- fractal_dimension1
- radius2
- texture2
- smoothness2
- compactness2
- fractal_dimension2
- radius3
- texture3
- area3
- smoothness3
- concavity3
- concave_points3
- symmetry3

Shape del Escenario 3 (LASSO): (569, 15)


## 4. Escenario 4: Selección No Lineal (Mutual Information)

**Justificación Teórica:**
Hasta ahora, hemos aplicado filtros lineales (Correlación de Pearson y LASSO). Estos métodos son excelentes para preparar los datos para un modelo lineal (como la Regresión Logística), pero podrían descartar variables que tienen relaciones curvas, complejas o no lineales con el diagnóstico de cáncer.

Para garantizar que nuestros modelos más abstractos (Redes Neuronales y Random Forest) reciban la información que necesitan, aplicaremos **Información Mutua (Mutual Information)**. A diferencia de ANOVA F-test (que es lineal y univariado) o Chi-Cuadrado (que es para variables categóricas), *Mutual Information* detecta cualquier tipo de dependencia no lineal. Seleccionaremos las **15 mejores características** para que compita en igualdad de condiciones contra las 15 de LASSO.

In [5]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

print("--- CREANDO ESCENARIO 4: INFORMACIÓN MUTUA (MUTUAL INFO) ---")

# 1. Configurar y entrenar el selector para las 15 mejores variables
selector_mi = SelectKBest(score_func=mutual_info_classif, k=15)
selector_mi.fit(X_scaled, df['Diagnosis'])

# 2. Obtener las columnas ganadoras
mascara_mi = selector_mi.get_support()
columnas_elegidas_mi = X_scaled.columns[mascara_mi]
X_mutual_info = X_scaled[columnas_elegidas_mi]

# 3. Imprimir con el formato solicitado
vars_descartadas_mi = len(X_scaled.columns) - len(columnas_elegidas_mi)
print(f"Variables descartadas por Mutual Information: {vars_descartadas_mi}")
print(f"Variables sobrevivientes ({len(columnas_elegidas_mi)}):")
for col in columnas_elegidas_mi:
    print(f"- {col}")

print(f"\nShape del Escenario 4 (Mutual Info): {X_mutual_info.shape}")

# 4. Exportar el escenario
ruta_mutual_info = './data/X_escenario_4_mutual_info.csv'
X_mutual_info.to_csv(ruta_mutual_info, index=False)

--- CREANDO ESCENARIO 4: INFORMACIÓN MUTUA (MUTUAL INFO) ---
Variables descartadas por Mutual Information: 15
Variables sobrevivientes (15):
- radius1
- perimeter1
- area1
- compactness1
- concavity1
- concave_points1
- radius2
- perimeter2
- area2
- radius3
- perimeter3
- area3
- compactness3
- concavity3
- concave_points3

Shape del Escenario 4 (Mutual Info): (569, 15)


## 5. Escenario 5: Método de Envoltura (Wrapper - RFE)

**Justificación Teórica:**
Para completar el espectro de técnicas de selección de características, implementaremos un Método de Envoltura (*Wrapper Method*). A diferencia de los Filtros (estadística pura) o los Incrustados (penalización matemática), los Wrappers utilizan un algoritmo predictivo real para evaluar iterativamente subconjuntos de características.

Utilizaremos la **Eliminación Recursiva de Características (RFE)** impulsada por una Regresión Logística. Para garantizar un "terreno de juego nivelado" en nuestra fase de evaluación final, forzaremos al algoritmo a seleccionar exactamente **15 variables**. Esto nos permitirá realizar una comparación justa y directa contra los Escenarios 3 (LASSO) y 4 (Mutual Information), determinando cuál de las tres filosofías de selección (Incrustado, Filtro o Wrapper) elige el mejor subconjunto dimensional para la predicción oncológica.

In [6]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

print("--- CREANDO ESCENARIO 5: WRAPPER (RFE) ---")

# 1. Definir el estimador base
estimador_base = LogisticRegression(solver='liblinear', random_state=30)

# 2. Configurar RFE para forzar exactamente 15 variables
selector_rfe = RFE(estimator=estimador_base, n_features_to_select=15, step=1)

# 3. Entrenar el selector con los datos estandarizados
selector_rfe.fit(X_scaled, df['Diagnosis'])

# 4. Obtener resultados
mascara_rfe = selector_rfe.support_
columnas_elegidas_rfe = X_scaled.columns[mascara_rfe]
X_wrapper_rfe = X_scaled[columnas_elegidas_rfe]

# 5. Imprimir con el formato solicitado
vars_descartadas_rfe = len(X_scaled.columns) - len(columnas_elegidas_rfe)
print(f"Variables descartadas por RFE: {vars_descartadas_rfe}")
print(f"Variables sobrevivientes ({len(columnas_elegidas_rfe)}):")
for col in columnas_elegidas_rfe:
    print(f"- {col}")

print(f"\nShape del Escenario 5 (RFE): {X_wrapper_rfe.shape}")

# 6. Exportar el escenario (¡Ojo con el nombre del archivo!)
ruta_rfe = './data/X_escenario_5_rfe.csv'
X_wrapper_rfe.to_csv(ruta_rfe, index=False)

--- CREANDO ESCENARIO 5: WRAPPER (RFE) ---
Variables descartadas por RFE: 15
Variables sobrevivientes (15):
- area1
- compactness1
- concave_points1
- radius2
- perimeter2
- area2
- compactness2
- radius3
- texture3
- perimeter3
- area3
- smoothness3
- concavity3
- concave_points3
- symmetry3

Shape del Escenario 5 (RFE): (569, 15)


## 6. Escenario 4: Reducción de Dimensionalidad (PCA)
A diferencia de los métodos anteriores, PCA no elimina variables, sino que las fusiona en un nuevo espacio geométrico. Extraeremos los Componentes Principales necesarios para encapsular el **95% de la información (varianza)** original de los tumores, eliminando la colinealidad al 100% pero perdiendo la interpretabilidad directa de las características.

In [7]:
# Inicializar PCA para conservar el 95% de la varianza
pca = PCA(n_components=0.95, random_state=30)

# Ajustar y transformar los datos
X_pca_array = pca.fit_transform(X_scaled)

# Crear nombres para los nuevos componentes (PC1, PC2, etc.)
nombres_pca = [f"PC{i+1}" for i in range(X_pca_array.shape[1])]
X_pca = pd.DataFrame(X_pca_array, columns=nombres_pca)

print(f"Componentes necesarios para explicar el 95% de la varianza: {pca.n_components_}")
print(f"Varianza total explicada: {sum(pca.explained_variance_ratio_) * 100:.2f}%\n")
print(f"Shape del Escenario 4 (PCA): {X_pca.shape}")

Componentes necesarios para explicar el 95% de la varianza: 10
Varianza total explicada: 95.16%

Shape del Escenario 4 (PCA): (569, 10)


## 7. Exportación de Escenarios
Finalmente, guardamos los 4 escenarios de características (junto con la variable objetivo `y`) en la carpeta `data/` para ser consumidos directamente por los algoritmos predictivos en la fase de **Estudio Comparativo (Notebook 03)**.

In [8]:
# Guardar los DataFrames transformados a CSV
X_escenario_1.to_csv('./data/X_escenario_1_crudo.csv', index=False)
X_manual.to_csv('./data/X_escenario_2_manual.csv', index=False)
X_lasso.to_csv('./data/X_escenario_3_lasso.csv', index=False)
X_mutual_info.to_csv('./data/X_escenario_4_mutual_info.csv', index=False)
X_wrapper_rfe.to_csv('./data/X_escenario_5_rfe.csv', index=False)
X_pca.to_csv('./data/X_escenario_6_pca.csv', index=False)
y.to_csv('./data/y_target.csv', index=False)

print("¡Exportación de los 6 escenarios completada con éxito!")

¡Exportación de los 6 escenarios completada con éxito!
